# 13 Gating Distribution Analysis

hard / soft gating 설정별로 어떤 timestep에서 distillation이 실제로 켜지는지 분포를 시각화합니다.

이 노트북은 모델 학습이 필요 없으며 단일 프로세스로 즉시 실행됩니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.experiments import load_cfg, run_gating_analysis

base_cfg = load_cfg("configs/cifar10.yaml")

gating_configs = {
    "hard (0.1–0.6)": {"sdd": {"gating": {"mode": "hard", "t_min": 0.1, "t_max": 0.6}}},
    "hard (0.2–0.8)": {"sdd": {"gating": {"mode": "hard", "t_min": 0.2, "t_max": 0.8}}},
    "soft β=0.08":    {"sdd": {"gating": {"mode": "soft", "t_min": 0.1, "t_max": 0.6, "soft_beta": 0.08}}},
    "soft β=0.04":    {"sdd": {"gating": {"mode": "soft", "t_min": 0.1, "t_max": 0.6, "soft_beta": 0.04}}},
}

results = run_gating_analysis(base_cfg, gating_configs=gating_configs)
print("수집 완료:", list(results.keys()))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (name, data) in zip(axes.flat, results.items()):
    t_norm = data["timesteps"] / data["T"]
    g = data["gate_values"]
    ax.scatter(t_norm, g, s=3, alpha=0.2, c="steelblue")
    ax.axvline(data["t_min"], color="red",   lw=1, ls="--", label=f"t_min={data['t_min']}")
    ax.axvline(data["t_max"], color="green", lw=1, ls="--", label=f"t_max={data['t_max']}")
    frac_on = float((g > 0.5).mean())
    ax.set_title(f"{name}\n{frac_on*100:.1f}% steps gated ON")
    ax.set_xlabel("Normalised timestep t/T"); ax.set_ylabel("Gate value")
    ax.set_ylim(-0.05, 1.15); ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle("Gating distribution by configuration", fontsize=14)
plt.tight_layout()
plt.savefig("outputs/figures/gating_analysis.png", dpi=150)
plt.show()


In [ ]:
import pandas as pd

rows = []
for name, data in results.items():
    g = data["gate_values"]
    rows.append({
        "config":          name,
        "mode":            data["mode"],
        "pct_fully_on":    f"{float((g > 0.99).mean())*100:.1f}%",
        "pct_partially_on":f"{float((g > 0.01).mean())*100:.1f}%",
        "mean_gate":       f"{float(g.mean()):.3f}",
    })
pd.DataFrame(rows)
